In [ ]:
%%writefile run_frozen_benchmark.py
from __future__ import annotations

"""Evaluate the six in-scope ASR models on evaluation/frozen_eval_manifest.csv.

Model downloads and GPU inference are deliberately opt-in via explicit CLI
invocation; this module preserves raw hypotheses and can be rerun independently
from scoring.
"""

import argparse
import csv
import gc
import json
import os
import re
import sys
import tempfile
import time
from collections import defaultdict
from pathlib import Path
from typing import Any

import numpy as np
import psutil
import soundfile as sf
import torch

from evaluate_asr import (
    aggregate_metrics,
    load_audio,
    make_telephony_audio,
    metric,
    NORMALIZATION_POLICY,
    resample,
    runtime_metadata,
    script_aware_metrics,
)

REPO_ROOT = Path(__file__).resolve().parents[1]
MODEL_METADATA = {
    "indicconformer": {
        "model_id": "ai4bharat/indic-conformer-600m-multilingual",
        "decoder": "CTC",
        "tokenizer_vocab_size": "checkpoint-defined; inspect model/tokenizer at runtime",
        "has_devanagari": "yes (Indic tokenizer)",
        "claims_hindi": "yes",
        "streaming": "chunked offline; 45-second windows",
        "long_audio": "45-second chunks, concatenated outputs",
    },
    "sraavani": {
        "model_id": "ARTPARK-IISc/SraVaani-1.0",
        "decoder": "hybrid TDT-CTC",
        "tokenizer_vocab_size": "checkpoint-defined; inspect model/tokenizer at runtime",
        "has_devanagari": "yes (Indic model)",
        "claims_hindi": "yes",
        "streaming": "chunked offline in this benchmark",
        "long_audio": "30-second windows with 1-second overlap",
    },
    "nemotron": {
        "model_id": "nvidia/nemotron-3.5-asr-streaming-0.6b",
        "decoder": "RNNT",
        "tokenizer_vocab_size": "checkpoint-defined; inspect processor tokenizer at runtime",
        "has_devanagari": "Hindi locale supported; Latin/Devanagari switching must be measured",
        "claims_hindi": "yes (hi-IN transcription-ready)",
        "streaming": "native cache-aware streaming; default lookahead 6 tokens / 560 ms",
        "long_audio": "streaming feature chunks with explicit terminal flush",
    },
    "parakeet_v2": {
        "model_id": "nvidia/parakeet-tdt-0.6b-v2",
        "decoder": "TDT",
        "tokenizer_vocab_size": "1024 (English vocabulary per evaluation brief)",
        "has_devanagari": "no (expected; record actual emissions)",
        "claims_hindi": "no; English-only release",
        "streaming": "offline transcription in this benchmark; NeMo chunked streaming is separate",
        "long_audio": "full-attention input supported to about 24 minutes per model card",
    },
    "parakeet_v3": {
        "model_id": "nvidia/parakeet-tdt-0.6b-v3",
        "decoder": "TDT",
        "tokenizer_vocab_size": "8192 (multilingual vocabulary per evaluation brief)",
        "has_devanagari": "no (Hindi absent per evaluation brief)",
        "claims_hindi": "no; not tuned for code-switching",
        "streaming": "chunked streaming supported via NeMo RNNT script; this benchmark uses offline transcription",
        "long_audio": "full attention to about 24 minutes; local attention for longer recordings",
    },
    "mms": {
        "model_id": "facebook/mms-1b-all",
        "decoder": "wav2vec2 CTC",
        "tokenizer_vocab_size": "Hindi adapter tokenizer; record runtime vocabulary size",
        "has_devanagari": "Hindi adapter-dependent",
        "claims_hindi": "yes (Hindi adapter)",
        "streaming": "not native in this benchmark; offline CTC",
        "long_audio": "40-second windows with overlap and text merge",
    },
}


def _torch_memory_reset(device: str) -> None:
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()


def _peak_memory_mb(device: str) -> float:
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 * 1024)
    info = psutil.Process().memory_info()
    return getattr(info, "peak_wset", info.rss) / (1024 * 1024)


def _model_name(model: Any, key: str) -> str:
    return str(getattr(model, "model_id", MODEL_METADATA[key]["model_id"]))


def _tokenizer_vocab_size(model: Any) -> int | str:
    candidates = [
        getattr(model, "tokenizer", None),
        getattr(getattr(model, "processor", None), "tokenizer", None),
    ]
    for tokenizer in candidates:
        if tokenizer is None:
            continue
        vocab_size = getattr(tokenizer, "vocab_size", None)
        if isinstance(vocab_size, int):
            return vocab_size
        try:
            return len(tokenizer)
        except (TypeError, AttributeError):
            pass
    config = getattr(model, "config", None)
    vocab_size = getattr(config, "vocab_size", None)
    return vocab_size if isinstance(vocab_size, int) else "checkpoint-defined; runtime API did not expose size"


def _make_adapter(key: str, device: str, language: str, lookahead_tokens: int):
    """Load one model; imports model-specific heavyweight packages lazily."""
    if key == "indicconformer":
        sys.path.insert(0, str(Path(__file__).resolve().parent))
        import indicconfor

        indicconfor.DEVICE = torch.device(device)
        model = indicconfor.build_model(MODEL_METADATA[key]["model_id"])
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            audio = resample(audio, sample_rate, 16000)
            return indicconfor.transcribe_long_audio(model, audio, language, 45.0)
        return model, transcribe, {
            "sample_rate_hz": 16000, "batch_size": 1,
            "tokenizer_vocab_size": _tokenizer_vocab_size(model),
        }

    if key == "sraavani":
        sys.path.insert(0, str(Path(__file__).resolve().parent))
        import sraavani

        sraavani.DEVICE = torch.device(device)
        model = sraavani.build_model(MODEL_METADATA[key]["model_id"])
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            audio = resample(audio, sample_rate, 16000)
            hypotheses, _, _ = sraavani.transcribe_batch(
                model=model,
                audio_list=[audio],
                model_batch_size=1,
                chunk_seconds=30.0,
                overlap_seconds=1.0,
                sample_rate=16000,
            )
            return hypotheses[0] if hypotheses else ""
        return model, transcribe, {
            "sample_rate_hz": 16000, "batch_size": 1,
            "chunk_seconds": 30.0, "overlap_seconds": 1.0,
            "tokenizer_vocab_size": _tokenizer_vocab_size(model),
        }

    if key == "nemotron":
        sys.path.insert(0, str(Path(__file__).resolve().parent))
        from transcriber_nemotron import NemotronTranscriber

        model = NemotronTranscriber(
            model_id=MODEL_METADATA[key]["model_id"],
            language=language,
            lookahead_tokens=lookahead_tokens,
            device=device,
        )
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            return model.transcribe(audio, sample_rate)[0]
        return model, transcribe, {
            "sample_rate_hz": model.sample_rate,
            "lookahead_tokens": lookahead_tokens,
            "streaming_latency_ms": getattr(model.processor, "streaming_latency_ms", None),
            "tokenizer_vocab_size": _tokenizer_vocab_size(model),
        }

    if key in {"parakeet_v2", "parakeet_v3"}:
        try:
            import nemo.collections.asr as nemo_asr
        except ImportError as exc:
            raise RuntimeError("Parakeet runs require NVIDIA NeMo ASR (nemo_toolkit[asr]).") from exc
        model = nemo_asr.models.ASRModel.from_pretrained(model_name=MODEL_METADATA[key]["model_id"])
        model = model.to(device).eval()
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            audio = resample(audio, sample_rate, 16000).astype(np.float32)
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as handle:
                wav_path = Path(handle.name)
            try:
                sf.write(str(wav_path), audio, 16000, subtype="PCM_16")
                hypotheses = model.transcribe(
                    [str(wav_path)], batch_size=1, return_hypotheses=True
                )
            finally:
                wav_path.unlink(missing_ok=True)
            if isinstance(hypotheses, tuple):
                hypotheses = hypotheses[0]
            first = hypotheses[0] if hypotheses else ""
            return str(getattr(first, "text", first)).strip()
        return model, transcribe, {
            "sample_rate_hz": 16000, "batch_size": 1,
            "tokenizer_vocab_size": _tokenizer_vocab_size(model),
        }

    if key == "mms":
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        model_id = MODEL_METADATA[key]["model_id"]
        processor = AutoProcessor.from_pretrained(model_id, target_lang="hin")
        model = Wav2Vec2ForCTC.from_pretrained(
            model_id, target_lang="hin", ignore_mismatched_sizes=True
        ).to(device).eval()
        chunk_seconds = 30.0
        overlap_seconds = 2.0
        sample_rate = int(processor.feature_extractor.sampling_rate)
        def transcribe(audio: np.ndarray, source_rate: int) -> str:
            audio = resample(audio, source_rate, sample_rate)
            chunk_size = int(chunk_seconds * sample_rate)
            step = int((chunk_seconds - overlap_seconds) * sample_rate)
            texts = []
            for start in range(0, len(audio), step):
                chunk = audio[start:start + chunk_size]
                if not len(chunk):
                    continue
                inputs = processor(
                    chunk, sampling_rate=sample_rate, return_tensors="pt", padding=True
                )
                inputs = {name: value.to(device) for name, value in inputs.items()}
                with torch.inference_mode():
                    logits = model(**inputs).logits
                token_ids = torch.argmax(logits, dim=-1)
                texts.append(processor.batch_decode(token_ids)[0].strip())
                if start + chunk_size >= len(audio):
                    break
            return _merge_overlap_texts(texts)
        return model, transcribe, {
            "sample_rate_hz": sample_rate,
            "tokenizer_vocab_size": len(processor.tokenizer),
            "batch_size": 1,
            "adapter": "hin",
            "chunk_seconds": chunk_seconds,
            "overlap_seconds": overlap_seconds,
        }

    raise ValueError(f"Unknown model key: {key}")


def _merge_overlap_texts(texts: list[str], max_overlap_words: int = 12) -> str:
    merged: list[str] = []
    for text in texts:
        current = text.split()
        overlap = 0
        for size in range(min(max_overlap_words, len(merged), len(current)), 0, -1):
            if [word.casefold() for word in merged[-size:]] == [
                word.casefold() for word in current[:size]
            ]:
                overlap = size
                break
        merged.extend(current[overlap:])
    return " ".join(merged).strip()


def _load_row_audio(row: dict[str, str]) -> tuple[np.ndarray, int]:
    audio, sample_rate = load_audio(Path(row["audio_path"]))
    start = float(row["start_seconds"])
    end = float(row["end_seconds"])
    begin_sample = max(0, int(round(start * sample_rate)))
    end_sample = min(len(audio), int(round(end * sample_rate)))
    audio = np.asarray(audio[begin_sample:end_sample], dtype=np.float32)
    if row["augmentation"] != "none":
        seed = int(row["augmentation_seed"])
        audio, sample_rate = make_telephony_audio(audio, sample_rate, seed)
    return audio, sample_rate


def evaluate_model(
    key: str,
    manifest_path: Path,
    output_path: Path,
    device: str,
    language: str,
    lookahead_tokens: int,
    threads: int | None,
) -> dict[str, Any]:
    if threads is not None:
        torch.set_num_threads(threads)
    model, transcribe, config = _make_adapter(key, device, language, lookahead_tokens)
    with manifest_path.open(encoding="utf-8-sig", newline="") as handle:
        manifest = list(csv.DictReader(handle))
    records_by_slice: dict[str, list[dict[str, Any]]] = defaultdict(list)
    run_meta = runtime_metadata(beam_size=None)
    run_meta["device"] = str(device)
    if device == "cpu":
        run_meta["gpu_name"] = None
    raw_path = output_path.with_suffix(".raw.jsonl")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with raw_path.open("w", encoding="utf-8", newline="\n") as raw_file:
        for row in manifest:
            audio, sample_rate = _load_row_audio(row)
            duration = len(audio) / sample_rate if sample_rate else 0.0
            _torch_memory_reset(device)
            started = time.perf_counter()
            hypothesis = transcribe(audio, sample_rate)
            if device == "cuda":
                torch.cuda.synchronize()
            elapsed = time.perf_counter() - started
            peak_memory = _peak_memory_mb(device)
            script_metrics = script_aware_metrics(row["reference"], hypothesis)
            record = {
                "utterance_id": row["utterance_id"],
                "audio": row["audio_path"],
                "recording_id": row["recording_id"],
                "recording_group_id": row["recording_group_id"],
                "speaker_id": row["speaker_id"],
                "slice": row["slice"],
                "reference": row["reference"],
                "reference_source": row["reference_source"],
                "reference_review_status": row["reference_review_status"],
                "hypothesis": hypothesis,
                "duration_seconds": duration,
                "inference_seconds": elapsed,
                "rtf": elapsed / max(duration, 1e-9),
                "peak_process_memory_mb": peak_memory,
                "device": run_meta["device"],
                "gpu_name": run_meta["gpu_name"],
                "threads": run_meta["threads"],
                "cpu_count": run_meta["cpu_count"],
                "beam_size": None,
                "metrics": metric(row["reference"], hypothesis),
                "script_aware": script_metrics,
                "augmentation": row["augmentation"],
                "augmentation_seed": row["augmentation_seed"],
            }
            raw_file.write(json.dumps(record, ensure_ascii=False) + "\n")
            raw_file.flush()
            records_by_slice[row["slice"]].append(record)
            print(f"[{key}][{row['slice']}] {row['utterance_id']} {elapsed:.3f}s")

    results: dict[str, Any] = {
        "model_key": key,
        **MODEL_METADATA[key],
        "device": run_meta["device"],
        "gpu_name": run_meta["gpu_name"],
        "threads": run_meta["threads"],
        "cpu_count": run_meta["cpu_count"],
        "beam_size": None,
        "manifest": str(manifest_path.resolve()),
        "raw_output_jsonl": str(raw_path.resolve()),
        "manifest_version": "saleagent-stt-eval-v1",
        "normalization": NORMALIZATION_POLICY,
        "model_config": config,
        "tokenizer_vocab_size": config.get(
            "tokenizer_vocab_size", MODEL_METADATA[key]["tokenizer_vocab_size"]
        ),
        "slices": {},
    }
    for slice_name, records in sorted(records_by_slice.items()):
        results["slices"][slice_name] = {
            "count": len(records),
            "aggregate": aggregate_metrics(records),
            "records": records,
        }
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", type=Path, default="/kaggle/working/frozen_eval_manifest.csv")
    parser.add_argument("--output-dir", type=Path, default= "/kaggle/working/modelbenchmark_day2")
    parser.add_argument("--models", nargs="+", choices=sorted(MODEL_METADATA), default=sorted(MODEL_METADATA))
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    parser.add_argument("--language", default="hi-IN")
    parser.add_argument("--lookahead-tokens", type=int, default=6)
    parser.add_argument("--threads", type=int, default=None)
    args = parser.parse_args()
    if not args.manifest.is_file():
        raise SystemExit(f"Evaluation manifest not found: {args.manifest}")
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else args.device
    if device == "auto":
        device = "cpu"
    if device == "cuda" and not torch.cuda.is_available():
        raise SystemExit("CUDA was requested but is unavailable.")

    errors = {}
    for key in args.models:
        output_path = args.output_dir / f"{key}_evaluation.json"
        try:
            evaluate_model(
                key, args.manifest, output_path, device, args.language,
                args.lookahead_tokens, args.threads,
            )
        except Exception as exc:
            errors[key] = f"{type(exc).__name__}: {exc}"
            print(f"[{key}] FAILED: {errors[key]}", file=sys.stderr)
    if errors:
        (args.output_dir / "run_errors.json").write_text(
            json.dumps(errors, indent=2), encoding="utf-8"
        )
        raise SystemExit(f"{len(errors)} model(s) failed; see {args.output_dir / 'run_errors.json'}")


if __name__ == "__main__":
    main()

In [9]:
import os
import pandas as pd

manifest_path = "/kaggle/working/frozen_eval_manifest.csv"
dataset_base = "/kaggle/input/datasets/nagababumolleti/salesagentdata"

# Step 1: Recursively locate and index every audio file in salesagentdata
file_lookup = {}
for root, dirs, files in os.walk(dataset_base):
  for f in files:
    file_lookup[f] = os.path.join(root, f)

# Step 2: Read CSV file
df = pd.read_csv(manifest_path)


# Helper function to convert Windows local path to exact Kaggle path
def fix_audio_path(old_path):
  if not isinstance(old_path, str):
    return old_path

  filename = old_path.replace("\\", "/").split("/")[-1]

  if filename in file_lookup:
    return file_lookup[filename]

  return old_path


# Step 3: Update only the 'audio_path' column
if "audio_path" in df.columns:
  df["audio_path"] = df["audio_path"].apply(fix_audio_path)

# Step 4: Save updated manifest
df.to_csv(manifest_path, index=False)
print("Updated audio_path in manifest successfully!")

Updated audio_path in manifest successfully!


In [10]:
import pandas as pd

df = pd.read_csv("/kaggle/working/frozen_eval_manifest.csv")
print(df["audio_path"].head())

0    /kaggle/input/datasets/nagababumolleti/salesag...
1    /kaggle/input/datasets/nagababumolleti/salesag...
2    /kaggle/input/datasets/nagababumolleti/salesag...
3    /kaggle/input/datasets/nagababumolleti/salesag...
4    /kaggle/input/datasets/nagababumolleti/salesag...
Name: audio_path, dtype: object


In [6]:
%%writefile run_frozen_benchmark.py
from __future__ import annotations

"""Evaluate in-scope ASR models on evaluation manifest.

All helper code for IndicConformer and SraVaani is embedded directly to ensure single-file portability.
"""

import argparse
import csv
import gc
import json
import os
import re
import sys
import tempfile
import time
from collections import defaultdict
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import psutil
import soundfile as sf
import torch

from evaluate_asr import (
    aggregate_metrics,
    load_audio,
    make_telephony_audio,
    metric,
    NORMALIZATION_POLICY,
    resample,
    runtime_metadata,
    script_aware_metrics,
)

# ==============================================================================
# EMBEDDED MODULE: IndicConformer Helper
# ==============================================================================
class EmbeddedIndicConformer:
    @staticmethod
    def build_model(model_id: str, device: torch.device):
        from transformers import AutoModel
        try:
            model = AutoModel.from_pretrained(model_id, trust_remote_code=True)
            model = model.to(device)
            model.eval()
            return model
        except OSError as error:
            message = str(error)
            if "gated repo" in message.lower() or "403" in message:
                raise SystemExit(
                    f"Hugging Face access is required for '{model_id}'. "
                    "Accept terms or authenticate via `huggingface-cli login`."
                ) from error
            raise

    @staticmethod
    def transcribe_long_audio(model, audio: np.ndarray, language: str, device: torch.device, chunk_seconds: float = 45.0) -> str:
        audio = np.asarray(audio, dtype=np.float32).flatten()
        chunk_samples = int(chunk_seconds * 16000)
        parts = []
        for start in range(0, len(audio), chunk_samples):
            chunk = audio[start:start + chunk_samples]
            if len(chunk) == 0:
                continue
            wav = torch.from_numpy(chunk).unsqueeze(0).to(device)
            result = model(wav, language, "ctc")
            if result is not None:
                parts.append(str(result).strip())
        return " ".join(part for part in parts if part)


# ==============================================================================
# EMBEDDED MODULE: SraVaani Helper
# ==============================================================================
@dataclass(frozen=True)
class SraVaaniChunkJob:
    file_index: int
    chunk_index: int
    audio: np.ndarray

class EmbeddedSraVaani:
    @staticmethod
    def build_model(model_id: str, device: torch.device):
        from huggingface_hub import snapshot_download
        from transformers import AutoModel
        hf_token = os.environ.get("HF_TOKEN")
        model_dir = snapshot_download(repo_id=model_id, token=hf_token)
        load_kwargs = {"trust_remote_code": True}
        if hf_token:
            load_kwargs["token"] = hf_token
        model = AutoModel.from_pretrained(model_dir, **load_kwargs)
        model = model.to(device)
        model.eval()
        return model

    @staticmethod
    def extract_hypothesis_text(hypothesis) -> str:
        if hypothesis is None:
            return ""
        if isinstance(hypothesis, str):
            return hypothesis.strip()
        if hasattr(hypothesis, "text"):
            return str(hypothesis.text).strip()
        if isinstance(hypothesis, (list, tuple)) and hypothesis:
            return EmbeddedSraVaani.extract_hypothesis_text(hypothesis[0])
        return str(hypothesis).strip()

    @staticmethod
    def merge_texts(previous: str, current: str, max_overlap_words: int = 12) -> str:
        previous, current = previous.strip(), current.strip()
        if not previous: return current
        if not current: return previous
        if previous == current: return previous
        prev_words, curr_words = previous.split(), current.split()
        overlap = 0
        for n in range(min(max_overlap_words, len(prev_words), len(curr_words)), 0, -1):
            if prev_words[-n:] == curr_words[:n]:
                overlap = n
                break
        if overlap:
            curr_words = curr_words[overlap:]
        return " ".join(prev_words + curr_words).strip()

    @staticmethod
    def transcribe_batch(model, audio_list: list[np.ndarray], device: torch.device, model_batch_size: int = 1, chunk_seconds: float = 30.0, overlap_seconds: float = 1.0):
        if not audio_list:
            return []
        
        chunk_samples = int(round(chunk_seconds * 16000))
        overlap_samples = int(round(overlap_seconds * 16000))
        step_samples = chunk_samples - overlap_samples
        
        jobs: list[SraVaaniChunkJob] = []
        for f_idx, audio in enumerate(audio_list):
            audio = np.asarray(audio, dtype=np.float32).flatten()
            if audio.size <= chunk_samples:
                jobs.append(SraVaaniChunkJob(f_idx, 0, audio))
            else:
                c_idx, start = 0, 0
                while start < audio.size:
                    end = min(start + chunk_samples, audio.size)
                    jobs.append(SraVaaniChunkJob(f_idx, c_idx, audio[start:end]))
                    c_idx += 1
                    if end >= audio.size: break
                    start += step_samples

        results = {}
        with tempfile.TemporaryDirectory(prefix="sravaani_chunks_") as temp_dir:
            temp_path = Path(temp_dir)
            for start_idx in range(0, len(jobs), model_batch_size):
                batch_jobs = jobs[start_idx:start_idx + model_batch_size]
                batch_paths = []
                for idx, job in enumerate(batch_jobs):
                    p = temp_path / f"job_{start_idx + idx}.wav"
                    sf.write(str(p), job.audio, 16000, subtype="PCM_16")
                    batch_paths.append(str(p))

                with torch.inference_mode():
                    hypotheses = model.transcribe(batch_paths, batch_size=len(batch_paths), return_hypotheses=True)
                if isinstance(hypotheses, tuple) and len(hypotheses) == 2:
                    hypotheses = hypotheses[0]
                if not isinstance(hypotheses, (list, tuple)):
                    hypotheses = [hypotheses]
                
                for job, hyp in zip(batch_jobs, hypotheses):
                    results[(job.file_index, job.chunk_index)] = EmbeddedSraVaani.extract_hypothesis_text(hyp)

        final_texts = []
        for f_idx in range(len(audio_list)):
            file_chunks = sorted([v for k, v in results.items() if k[0] == f_idx], key=lambda x: x[0])
            merged = ""
            for _, text in file_chunks:
                merged = EmbeddedSraVaani.merge_texts(merged, text)
            final_texts.append(merged)
        return final_texts


# ==============================================================================
# MAIN BENCHMARK ENGINE
# ==============================================================================
MODEL_METADATA = {
    "indicconformer": {
        "model_id": "ai4bharat/indic-conformer-600m-multilingual",
        "decoder": "CTC",
        "has_devanagari": "yes (Indic tokenizer)",
        "claims_hindi": "yes",
    },
    "sraavani": {
        "model_id": "ARTPARK-IISc/SraVaani-1.0",
        "decoder": "hybrid TDT-CTC",
        "has_devanagari": "yes (Indic model)",
        "claims_hindi": "yes",
    },
    "nemotron": {
        "model_id": "nvidia/nemotron-3.5-asr-streaming-0.6b",
        "decoder": "RNNT",
        "has_devanagari": "Hindi locale supported",
        "claims_hindi": "yes",
    },
    "parakeet_v2": {
        "model_id": "nvidia/parakeet-tdt-0.6b-v2",
        "decoder": "TDT",
        "has_devanagari": "no",
        "claims_hindi": "no",
    },
    "parakeet_v3": {
        "model_id": "nvidia/parakeet-tdt-0.6b-v3",
        "decoder": "TDT",
        "has_devanagari": "no",
        "claims_hindi": "no",
    },
    "mms": {
        "model_id": "facebook/mms-1b-all",
        "decoder": "wav2vec2 CTC",
        "has_devanagari": "Hindi adapter-dependent",
        "claims_hindi": "yes",
    },
}

def _torch_memory_reset(device: str) -> None:
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        torch.cuda.reset_peak_memory_stats()

def _peak_memory_mb(device: str) -> float:
    if device == "cuda" and torch.cuda.is_available():
        torch.cuda.synchronize()
        return torch.cuda.max_memory_allocated() / (1024 * 1024)
    info = psutil.Process().memory_info()
    return getattr(info, "peak_wset", info.rss) / (1024 * 1024)

def _tokenizer_vocab_size(model: Any) -> int | str:
    candidates = [
        getattr(model, "tokenizer", None),
        getattr(getattr(model, "processor", None), "tokenizer", None),
    ]
    for tokenizer in candidates:
        if tokenizer is None: continue
        vocab_size = getattr(tokenizer, "vocab_size", None)
        if isinstance(vocab_size, int): return vocab_size
        try: return len(tokenizer)
        except (TypeError, AttributeError): pass
    config = getattr(model, "config", None)
    vocab_size = getattr(config, "vocab_size", None)
    return vocab_size if isinstance(vocab_size, int) else "checkpoint-defined"

def _make_adapter(key: str, device: str, language: str, lookahead_tokens: int):
    device_obj = torch.device(device)

    if key == "indicconformer":
        model = EmbeddedIndicConformer.build_model(MODEL_METADATA[key]["model_id"], device_obj)
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            audio = resample(audio, sample_rate, 16000)
            return EmbeddedIndicConformer.transcribe_long_audio(model, audio, language, device_obj, 45.0)
        return model, transcribe, {"sample_rate_hz": 16000, "batch_size": 1}

    if key == "sraavani":
        model = EmbeddedSraVaani.build_model(MODEL_METADATA[key]["model_id"], device_obj)
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            audio = resample(audio, sample_rate, 16000)
            hypotheses = EmbeddedSraVaani.transcribe_batch(model, [audio], device_obj)
            return hypotheses[0] if hypotheses else ""
        return model, transcribe, {"sample_rate_hz": 16000, "batch_size": 1}

    if key == "nemotron":
        try:
            from transcriber_nemotron import NemotronTranscriber
        except ImportError:
            raise RuntimeError("Nemotron evaluation requires transcriber_nemotron.py module.")
        model = NemotronTranscriber(
            model_id=MODEL_METADATA[key]["model_id"],
            language=language,
            lookahead_tokens=lookahead_tokens,
            device=device,
        )
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            return model.transcribe(audio, sample_rate)[0]
        return model, transcribe, {"sample_rate_hz": model.sample_rate}

    if key in {"parakeet_v2", "parakeet_v3"}:
        try:
            import nemo.collections.asr as nemo_asr
        except ImportError as exc:
            raise RuntimeError("Parakeet runs require NVIDIA NeMo ASR (nemo_toolkit[asr]).") from exc
        model = nemo_asr.models.ASRModel.from_pretrained(model_name=MODEL_METADATA[key]["model_id"]).to(device).eval()
        def transcribe(audio: np.ndarray, sample_rate: int) -> str:
            audio = resample(audio, sample_rate, 16000).astype(np.float32)
            with tempfile.NamedTemporaryFile(suffix=".wav", delete=False) as handle:
                wav_path = Path(handle.name)
            try:
                sf.write(str(wav_path), audio, 16000, subtype="PCM_16")
                hypotheses = model.transcribe([str(wav_path)], batch_size=1, return_hypotheses=True)
            finally:
                wav_path.unlink(missing_ok=True)
            if isinstance(hypotheses, tuple): hypotheses = hypotheses[0]
            first = hypotheses[0] if hypotheses else ""
            return str(getattr(first, "text", first)).strip()
        return model, transcribe, {"sample_rate_hz": 16000}

    if key == "mms":
        from transformers import AutoProcessor, Wav2Vec2ForCTC
        model_id = MODEL_METADATA[key]["model_id"]
        processor = AutoProcessor.from_pretrained(model_id, target_lang="hin")
        model = Wav2Vec2ForCTC.from_pretrained(
            model_id, target_lang="hin", ignore_mismatched_sizes=True
        ).to(device).eval()
        sample_rate = int(float(processor.feature_extractor.sampling_rate))
        
        def transcribe(audio: np.ndarray, source_rate: int) -> str:
            audio = resample(audio, source_rate, sample_rate)
            chunk_size = int(30.0 * sample_rate)
            step = int(28.0 * sample_rate)
            texts = []
            for start in range(0, len(audio), step):
                chunk = audio[start:start + chunk_size]
                if not len(chunk): continue
                inputs = processor(chunk, sampling_rate=sample_rate, return_tensors="pt", padding=True)
                inputs = {name: value.to(device) for name, value in inputs.items()}
                with torch.inference_mode():
                    logits = model(**inputs).logits
                token_ids = torch.argmax(logits, dim=-1)
                texts.append(processor.batch_decode(token_ids)[0].strip())
                if start + chunk_size >= len(audio): break
            return " ".join(texts).strip()
        return model, transcribe, {"sample_rate_hz": sample_rate}

    raise ValueError(f"Unknown model key: {key}")


def _load_row_audio(row: dict[str, str]) -> tuple[np.ndarray, int]:
    audio, sample_rate = load_audio(Path(row["audio_path"]))
    start = float(row.get("start_seconds", 0.0))
    end = float(row.get("end_seconds", len(audio) / sample_rate))
    begin_sample = max(0, int(round(start * sample_rate)))
    end_sample = min(len(audio), int(round(end * sample_rate)))
    audio = np.asarray(audio[begin_sample:end_sample], dtype=np.float32)
    
    aug = str(row.get("augmentation", "none")).lower()
    if aug != "none" and aug != "nan":
        # Safe float-to-int conversion fix for seed and sample rates
        seed_val = row.get("augmentation_seed", 0)
        seed = int(float(seed_val)) if str(seed_val).replace('.', '', 1).isdigit() else 0
        audio, sample_rate = make_telephony_audio(audio, sample_rate, seed)
    return audio, sample_rate


def evaluate_model(key: str, manifest_path: Path, output_path: Path, device: str, language: str, lookahead_tokens: int, threads: int | None) -> dict[str, Any]:
    if threads is not None:
        torch.set_num_threads(threads)
    model, transcribe, config = _make_adapter(key, device, language, lookahead_tokens)
    with manifest_path.open(encoding="utf-8-sig", newline="") as handle:
        manifest = list(csv.DictReader(handle))
    
    records_by_slice: dict[str, list[dict[str, Any]]] = defaultdict(list)
    run_meta = runtime_metadata(beam_size=None)
    run_meta["device"] = str(device)
    raw_path = output_path.with_suffix(".raw.jsonl")
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with raw_path.open("w", encoding="utf-8", newline="\n") as raw_file:
        for row in manifest:
            try:
                audio, sample_rate = _load_row_audio(row)
                duration = len(audio) / sample_rate if sample_rate else 0.0
                _torch_memory_reset(device)
                started = time.perf_counter()
                hypothesis = transcribe(audio, sample_rate)
                if device == "cuda":
                    torch.cuda.synchronize()
                elapsed = time.perf_counter() - started
                peak_memory = _peak_memory_mb(device)
                
                record = {
                    "utterance_id": row.get("utterance_id", ""),
                    "audio": row.get("audio_path", ""),
                    "slice": row.get("slice", "default"),
                    "reference": row.get("reference", ""),
                    "hypothesis": hypothesis,
                    "duration_seconds": duration,
                    "inference_seconds": elapsed,
                    "rtf": elapsed / max(duration, 1e-9),
                    "peak_process_memory_mb": peak_memory,
                    "metrics": metric(row.get("reference", ""), hypothesis),
                    "script_aware": script_aware_metrics(row.get("reference", ""), hypothesis),
                }
                raw_file.write(json.dumps(record, ensure_ascii=False) + "\n")
                raw_file.flush()
                records_by_slice[row.get("slice", "default")].append(record)
                print(f"[{key}][{row.get('slice', 'default')}] {row.get('utterance_id')} {elapsed:.3f}s")
            except Exception as row_exc:
                print(f"[{key}] SKIPPED row {row.get('utterance_id')}: {row_exc}", file=sys.stderr)

    results: dict[str, Any] = {
        "model_key": key,
        **MODEL_METADATA[key],
        "manifest": str(manifest_path.resolve()),
        "raw_output_jsonl": str(raw_path.resolve()),
        "slices": {},
    }
    for slice_name, records in sorted(records_by_slice.items()):
        results["slices"][slice_name] = {
            "count": len(records),
            "aggregate": aggregate_metrics(records),
            "records": records,
        }
    output_path.write_text(json.dumps(results, ensure_ascii=False, indent=2), encoding="utf-8")
    del model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return results


def main() -> None:
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("--manifest", type=Path, default="/kaggle/working/frozen_eval_manifest.csv")
    parser.add_argument("--output-dir", type=Path, default="/kaggle/working/modelbenchmark_day2")
    parser.add_argument("--models", nargs="+", choices=sorted(MODEL_METADATA), default=sorted(MODEL_METADATA))
    parser.add_argument("--device", choices=("auto", "cpu", "cuda"), default="auto")
    parser.add_argument("--language", default="hi-IN")
    parser.add_argument("--lookahead-tokens", type=int, default=6)
    parser.add_argument("--threads", type=int, default=None)
    args = parser.parse_args()

    if not args.manifest.is_file():
        raise SystemExit(f"Evaluation manifest not found: {args.manifest}")
    device = "cuda" if args.device == "auto" and torch.cuda.is_available() else args.device
    if device == "auto":
        device = "cpu"

    errors = {}
    for key in args.models:
        output_path = args.output_dir / f"{key}_evaluation.json"
        try:
            evaluate_model(key, args.manifest, output_path, device, args.language, args.lookahead_tokens, args.threads)
        except Exception as exc:
            errors[key] = f"{type(exc).__name__}: {exc}"
            print(f"[{key}] FAILED: {errors[key]}", file=sys.stderr)
            
    if errors:
        (args.output_dir / "run_errors.json").write_text(json.dumps(errors, indent=2), encoding="utf-8")


if __name__ == "__main__":
    main()

Writing run_frozen_benchmark.py


In [ ]:
import os

os.environ["HF_TOKEN"] =

In [5]:
!pip install onnxruntime

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 67.3 MB/s eta 0:00:00:00:0100:01


In [6]:
!pip install -q soundfile psutil pandas numpy transformers datasets torch torchaudio

In [7]:
!pip install -q onnxruntime huggingface_hub librosa SentencePiece

In [8]:
%%writefile evaluate_asr.py
from __future__ import annotations

import json
import os
import platform
import re
import unicodedata
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import soundfile as sf


AUDIO_EXTENSIONS = {".wav", ".mp3", ".flac", ".m4a", ".ogg", ".aac"}
DEVANAGARI_RE = re.compile(r"[\u0900-\u097f]")
LATIN_WORD_RE = re.compile(r"[A-Za-z]+(?:['-][A-Za-z]+)*")
NUMBER_RE = re.compile(r"\d+(?:[.,]\d+)*")
NORMALIZATION_POLICY = (
    "casefold; strip zero-width characters; remove punctuation including danda; "
    "remove separators inside digit groups; collapse whitespace; preserve "
    "Devanagari combining marks and nukta distinctions"
)


@dataclass
class Example:
    audio: Path
    reference: str


def normalize_text(text: str) -> str:
    """Normalize text identically for all ASR scoring paths.

    Removes zero-width characters and punctuation (including danda), strips
    separators inside digit groups, case-folds Latin text, and preserves
    Devanagari combining marks and nukta distinctions.
    """
    text = text.casefold().replace("\u200c", "").replace("\u200d", "")
    text = NUMBER_RE.sub(lambda match: re.sub(r"[.,]", "", match.group()), text)
    normalized = []
    for char in text:
        category = unicodedata.category(char)
        if char.isspace():
            normalized.append(" ")
        elif category.startswith(("P", "S")):
            normalized.append(" ")
        elif category.startswith("M") and "\u0900" <= char <= "\u097f":
            normalized.append(char)
        elif category[0] in {"L", "N"} or "\u0900" <= char <= "\u097f":
            normalized.append(char)
        else:
            normalized.append(" ")
    return " ".join("".join(normalized).split())


def words(text: str) -> list[str]:
    return normalize_text(text).split()


def characters(text: str) -> list[str]:
    return [char for char in normalize_text(text) if not char.isspace()]


def edit_distance(reference: list[str], hypothesis: list[str]) -> int:
    previous = list(range(len(hypothesis) + 1))
    for ref_index, ref_token in enumerate(reference, 1):
        current = [ref_index]
        for hyp_index, hyp_token in enumerate(hypothesis, 1):
            current.append(
                min(
                    current[-1] + 1,
                    previous[hyp_index] + 1,
                    previous[hyp_index - 1] + (ref_token != hyp_token),
                )
            )
        previous = current
    return previous[-1]


def error_rate(reference: list[str], hypothesis: list[str]) -> float | None:
    return None if not reference else edit_distance(reference, hypothesis) / len(reference)


def align_words(reference: list[str], hypothesis: list[str]) -> list[tuple[str | None, str | None]]:
    """Return a minimum-edit alignment, with deterministic tie handling."""
    rows, columns = len(reference) + 1, len(hypothesis) + 1
    costs = [[0] * columns for _ in range(rows)]
    operations = [[""] * columns for _ in range(rows)]
    for i in range(1, rows):
        costs[i][0], operations[i][0] = i, "delete"
    for j in range(1, columns):
        costs[0][j], operations[0][j] = j, "insert"

    for i in range(1, rows):
        for j in range(1, columns):
            substitution = costs[i - 1][j - 1] + (reference[i - 1] != hypothesis[j - 1])
            deletion = costs[i - 1][j] + 1
            insertion = costs[i][j - 1] + 1
            # Prefer diagonal (match/substitution), then deletion, then insertion.
            best = min(substitution, deletion, insertion)
            costs[i][j] = best
            operations[i][j] = (
                "equal" if reference[i - 1] == hypothesis[j - 1] else "substitute"
            ) if substitution == best else "delete" if deletion == best else "insert"

    aligned: list[tuple[str | None, str | None]] = []
    i, j = len(reference), len(hypothesis)
    while i or j:
        operation = operations[i][j]
        if operation in {"equal", "substitute"}:
            aligned.append((reference[i - 1], hypothesis[j - 1]))
            i -= 1
            j -= 1
        elif operation == "delete":
            aligned.append((reference[i - 1], None))
            i -= 1
        else:
            aligned.append((None, hypothesis[j - 1]))
            j -= 1
    aligned.reverse()
    return aligned


def _is_latin_word(token: str) -> bool:
    return bool(LATIN_WORD_RE.fullmatch(token))


def _has_devanagari(token: str) -> bool:
    return bool(DEVANAGARI_RE.search(token))


def runtime_metadata(beam_size: int | str | None = None) -> dict[str, Any]:
    """Return comparable hardware/decoder metadata for an inference run."""
    try:
        import torch

        device = "cuda" if torch.cuda.is_available() else "cpu"
        torch_threads = torch.get_num_threads()
        gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else None
    except (ImportError, RuntimeError):
        device, torch_threads, gpu_name = "cpu", None, None
    return {
        "device": device,
        "gpu_name": gpu_name,
        "threads": torch_threads or os.cpu_count(),
        "cpu_count": os.cpu_count(),
        "beam_size": beam_size,
        "platform": platform.platform(),
    }


def metric(reference: str, hypothesis: str) -> dict[str, float | int | None]:
    ref_words, hyp_words = words(reference), words(hypothesis)
    ref_chars, hyp_chars = characters(reference), characters(hypothesis)
    return {
        "wer": error_rate(ref_words, hyp_words),
        "cer": error_rate(ref_chars, hyp_chars),
        "reference_words": len(ref_words),
        "reference_characters": len(ref_chars),
    }


def aggregate_metrics(records: list[dict[str, Any]]) -> dict[str, float | int | None]:
    word_errors = sum(edit_distance(words(item["reference"]), words(item["hypothesis"])) for item in records)
    character_errors = sum(
        edit_distance(characters(item["reference"]), characters(item["hypothesis"])) for item in records
    )
    reference_words = sum(len(words(item["reference"])) for item in records)
    reference_characters = sum(len(characters(item["reference"])) for item in records)
    duration_seconds = sum(float(item.get("duration_seconds", 0.0)) for item in records)
    elapsed_seconds = 0.0
    timed_record_count = 0
    batch_elapsed: dict[str, float] = {}
    for index, item in enumerate(records):
        inference_seconds = item.get("inference_seconds")
        if inference_seconds is not None:
            elapsed_seconds += float(inference_seconds)
            timed_record_count += 1
        elif item.get("batch_elapsed_seconds") is not None:
            batch_id = str(item.get("batch_id", index))
            batch_elapsed[batch_id] = float(item["batch_elapsed_seconds"])
        elif item.get("rtf") is not None:
            # Backward compatibility for archived raw outputs; do not use this
            # reconstructed timing for newly produced batched evaluations.
            elapsed_seconds += float(item["rtf"]) * float(item.get("duration_seconds", 0.0))
            timed_record_count += 1
    elapsed_seconds += sum(batch_elapsed.values())
    script_totals = {
        "english_reference_words": 0,
        "english_correct_words": 0,
        "aligned_reference_words": 0,
        "wrong_script_words": 0,
        "english_as_devanagari": 0,
        "hindi_as_latin": 0,
    }
    for item in records:
        script = item.get("script_aware", {})
        for key in script_totals:
            script_totals[key] += int(script.get(key, 0) or 0)
    return {
        "wer": word_errors / reference_words if reference_words else None,
        "cer": character_errors / reference_characters if reference_characters else None,
        "word_errors": word_errors,
        "character_errors": character_errors,
        "reference_words": reference_words,
        "reference_characters": reference_characters,
        "inference_seconds": elapsed_seconds,
        "duration_seconds": duration_seconds,
        "mean_rtf": elapsed_seconds / duration_seconds if duration_seconds else None,
        "script_aware": {
            **script_totals,
            "english_word_accuracy": (
                script_totals["english_correct_words"] / script_totals["english_reference_words"]
                if script_totals["english_reference_words"] else None
            ),
            "wrong_script_rate": (
                script_totals["wrong_script_words"] / script_totals["aligned_reference_words"]
                if script_totals["aligned_reference_words"] else None
            ),
        },
        "max_peak_process_memory_mb": max(
            (item["peak_process_memory_mb"] for item in records
             if item.get("peak_process_memory_mb") is not None),
            default=None,
        ),
        "timing_mode": (
            "batch_wall_time_pooled_once" if batch_elapsed else
            "per_utterance_sum" if timed_record_count else "unavailable"
        ),
    }


def script_aware_metrics(reference: str, hypothesis: str) -> dict[str, Any]:
    ref_words = words(reference)
    hyp_words = words(hypothesis)
    aligned = align_words(ref_words, hyp_words)
    ref_devanagari = [char for char in characters(reference) if DEVANAGARI_RE.match(char)]
    hyp_devanagari = [char for char in characters(hypothesis) if DEVANAGARI_RE.match(char)]
    ref_latin = LATIN_WORD_RE.findall(normalize_text(reference))
    hyp_latin = LATIN_WORD_RE.findall(normalize_text(hypothesis))

    english_reference_words = 0
    english_correct_words = 0
    aligned_reference_words = 0
    english_as_devanagari = 0
    hindi_as_latin = 0
    for ref_token, hyp_token in aligned:
        if ref_token is None:
            continue
        is_english = _is_latin_word(ref_token)
        is_hindi = _has_devanagari(ref_token)
        if is_english:
            english_reference_words += 1
            if hyp_token == ref_token and hyp_token is not None and _is_latin_word(hyp_token):
                english_correct_words += 1
        if hyp_token is None:
            continue
        if is_english or is_hindi:
            aligned_reference_words += 1
        if is_english and _has_devanagari(hyp_token):
            english_as_devanagari += 1
        elif is_hindi and _is_latin_word(hyp_token):
            hindi_as_latin += 1
    wrong_script_words = english_as_devanagari + hindi_as_latin

    return {
        "is_code_mixed": bool(ref_devanagari and ref_latin),
        "mixed_error_rate": {
            "devanagari_cer": error_rate(ref_devanagari, hyp_devanagari),
            "latin_word_wer": error_rate(ref_latin, hyp_latin),
        },
        "english_word_accuracy": (
            english_correct_words / english_reference_words
            if english_reference_words
            else None
        ),
        "english_reference_words": english_reference_words,
        "english_correct_words": english_correct_words,
        "aligned_reference_words": aligned_reference_words,
        "wrong_script_words": wrong_script_words,
        "english_as_devanagari": english_as_devanagari,
        "hindi_as_latin": hindi_as_latin,
        "wrong_script_rate": (
            wrong_script_words / aligned_reference_words
            if aligned_reference_words else None
        ),
    }


def read_manifest(path: Path) -> list[Example]:
    records = json.loads(path.read_text(encoding="utf-8"))
    return [Example(Path(item["audio"]), item["text"]) for item in records]


def read_ground_truth(path: Path) -> list[str]:
    return [line.strip() for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]


def read_audio_mappings(path: Path) -> dict[Path, str]:
    mappings = {}
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "\t" not in line:
            continue
        audio_path, identifier = line.split("\t", maxsplit=1)
        mappings[(path.parent / audio_path).resolve()] = identifier.strip()
    return mappings


def find_examples(data_dir: Path, ground_truth_path: Path) -> list[Example]:
    lines = read_ground_truth(ground_truth_path)
    mappings = read_audio_mappings(ground_truth_path)
    audio_files = sorted(
        path for path in data_dir.rglob("*") if path.is_file() and path.suffix.lower() in AUDIO_EXTENSIONS
    )
    examples = []
    for audio in audio_files:
        mapped_identifier = mappings.get(audio.resolve())
        candidates = [mapped_identifier] if mapped_identifier else [audio.stem, audio.parent.name]
        matching_segments = []
        for line in lines:
            parts = line.split(maxsplit=1)
            if len(parts) != 2:
                continue
            identifier = parts[0].rstrip(":")
            if any(
                identifier == candidate
                or (candidate.endswith("_") and identifier.startswith(candidate))
                or identifier.startswith(f"{candidate}_")
                or f"_{candidate}_" in identifier
                for candidate in candidates
            ):
                matching_segments.append(parts[1])
        reference = " ".join(matching_segments) if matching_segments else None
        if reference is None:
            reference = next((line for line in lines if line.startswith(f"{audio.stem}:")), None)
        if reference:
            examples.append(Example(audio, reference))
    return examples


def resample(audio: np.ndarray, source_rate: int, target_rate: int) -> np.ndarray:
    if source_rate == target_rate:
        return audio
    target_length = max(1, round(len(audio) * target_rate / source_rate))
    old_positions = np.linspace(0, 1, len(audio), endpoint=False)
    new_positions = np.linspace(0, 1, target_length, endpoint=False)
    return np.interp(new_positions, old_positions, audio).astype(np.float32)


def make_telephony_audio(audio: np.ndarray, sample_rate: int, seed: int) -> tuple[np.ndarray, int]:
    rng = np.random.default_rng(seed)
    narrowband = resample(audio, sample_rate, 8000)
    mu = 255.0
    encoded = np.sign(narrowband) * np.log1p(mu * np.abs(narrowband)) / np.log1p(mu)
    decoded = np.sign(encoded) * (np.expm1(np.abs(encoded) * np.log1p(mu)) / mu)
    noisy = decoded + rng.normal(0, 0.008, len(decoded)).astype(np.float32)
    return np.clip(resample(noisy, 8000, sample_rate), -1, 1), sample_rate


def load_audio(path: Path) -> tuple[np.ndarray, int]:
    audio, sample_rate = sf.read(path, dtype="float32", always_2d=False)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    return audio, sample_rate




Writing evaluate_asr.py


In [15]:
%%writefile run_sravaani_only.py
import os
import json
import tempfile
import numpy as np
import pandas as pd
import soundfile as sf
import torch
from pathlib import Path
from dataclasses import dataclass
from transformers import AutoModel
from huggingface_hub import snapshot_download

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
MODEL_ID = "ARTPARK-IISc/SraVaani-1.0"
MODEL_SAMPLE_RATE = 16000

@dataclass(frozen=True)
class SraVaaniChunkJob:
    file_index: int
    chunk_index: int
    audio: np.ndarray

def safe_load_audio(path: str) -> tuple[np.ndarray, int]:
    """Safely loads audio files regardless of tuple return lengths."""
    data, sr = sf.read(path, dtype="float32", always_2d=False)
    if data.ndim == 2:
        data = data.mean(axis=1)
    return np.asarray(data, dtype=np.float32).flatten(), sr

def resample(audio: np.ndarray, orig_sr: int, target_sr: int) -> np.ndarray:
    if orig_sr == target_sr:
        return audio
    import librosa
    return librosa.resample(audio, orig_sr=orig_sr, target_sr=target_sr)

def build_sravaani_model(model_id: str):
    hf_token = os.environ.get("HF_TOKEN")
    print(f"Downloading/loading SraVaani model on {DEVICE}...")
    model_dir = snapshot_download(repo_id=model_id, token=hf_token)
    load_kwargs = {"trust_remote_code": True}
    if hf_token:
        load_kwargs["token"] = hf_token
    model = AutoModel.from_pretrained(model_dir, **load_kwargs)
    return model.to(DEVICE).eval()

def extract_text(hypothesis) -> str:
    if hypothesis is None:
        return ""
    if isinstance(hypothesis, str):
        return hypothesis.strip()
    if hasattr(hypothesis, "text"):
        return str(hypothesis.text).strip()
    if isinstance(hypothesis, (list, tuple)) and hypothesis:
        return extract_text(hypothesis[0])
    return str(hypothesis).strip()

def merge_texts(prev: str, curr: str, max_overlap_words: int = 12) -> str:
    prev, curr = prev.strip(), curr.strip()
    if not prev: return curr
    if not curr: return prev
    if prev == curr: return prev
    p_words, c_words = prev.split(), curr.split()
    overlap = 0
    for n in range(min(max_overlap_words, len(p_words), len(c_words)), 0, -1):
        if p_words[-n:] == c_words[:n]:
            overlap = n
            break
    if overlap:
        c_words = c_words[overlap:]
    return " ".join(p_words + c_words).strip()

def transcribe_batch_sravaani(model, audio_list: list[np.ndarray], chunk_seconds: float = 30.0, overlap_seconds: float = 1.0):
    if not audio_list:
        return []
    
    chunk_samples = int(round(chunk_seconds * MODEL_SAMPLE_RATE))
    overlap_samples = int(round(overlap_seconds * MODEL_SAMPLE_RATE))
    step_samples = chunk_samples - overlap_samples
    
    jobs: list[SraVaaniChunkJob] = []
    for f_idx, audio in enumerate(audio_list):
        audio = np.asarray(audio, dtype=np.float32).flatten()
        if audio.size <= chunk_samples:
            jobs.append(SraVaaniChunkJob(f_idx, 0, audio))
        else:
            c_idx, start = 0, 0
            while start < audio.size:
                end = min(start + chunk_samples, audio.size)
                jobs.append(SraVaaniChunkJob(f_idx, c_idx, audio[start:end]))
                c_idx += 1
                if end >= audio.size: break
                start += step_samples

    results = {}
    with tempfile.TemporaryDirectory(prefix="sravaani_chunks_") as temp_dir:
        temp_path = Path(temp_dir)
        batch_paths = []
        for idx, job in enumerate(jobs):
            p = temp_path / f"job_{idx}.wav"
            sf.write(str(p), job.audio, MODEL_SAMPLE_RATE, subtype="PCM_16")
            batch_paths.append(str(p))

        with torch.inference_mode():
            hypotheses = model.transcribe(batch_paths, batch_size=len(batch_paths), return_hypotheses=True)
        if isinstance(hypotheses, tuple) and len(hypotheses) == 2:
            hypotheses = hypotheses[0]
        if not isinstance(hypotheses, (list, tuple)):
            hypotheses = [hypotheses]
        
        for job, hyp in zip(jobs, hypotheses):
            results[(job.file_index, job.chunk_index)] = extract_text(hyp)

    final_texts = []
    for f_idx in range(len(audio_list)):
        file_chunks = sorted([v for k, v in results.items() if k[0] == f_idx], key=lambda x: x[0])
        merged = ""
        for _, text in file_chunks:
            merged = merge_texts(merged, text)
        final_texts.append(merged)
    return final_texts

def process_folder(dataset_dir: str, manifest_path: str = None, output_path: str = "/kaggle/working/sravaani_results.jsonl"):
    model = build_sravaani_model(MODEL_ID)
    
    # Process using manifest CSV if provided
    if manifest_path and os.path.exists(manifest_path):
        print(f"Processing manifest: {manifest_path}")
        df = pd.read_csv(manifest_path)
        
        with open(output_path, "w", encoding="utf-8") as out_f:
            for idx, row in df.iterrows():
                audio_path = row["audio_path"]
                utt_id = row.get("utterance_id", f"utt_{idx}")
                try:
                    audio, sr = safe_load_audio(audio_path)
                    audio = resample(audio, sr, MODEL_SAMPLE_RATE)
                    hyp = transcribe_batch_sravaani(model, [audio])[0]
                    
                    res = {
                        "utterance_id": utt_id,
                        "audio_path": audio_path,
                        "hypothesis": hyp,
                        "reference": row.get("reference", "")
                    }
                    out_f.write(json.dumps(res, ensure_ascii=False) + "\n")
                    print(f"[SraVaani] {utt_id}: {hyp}")
                except Exception as e:
                    print(f"[SraVaani] Error on {utt_id}: {e}")
                    
    # Process directory recursively if no manifest is provided
    else:
        print(f"Scanning directory recursively: {dataset_dir}")
        audio_files = []
        for root, _, files in os.walk(dataset_dir):
            for f in files:
                if f.lower().endswith((".mp3", ".wav", ".flac", ".m4a")):
                    audio_files.append(os.path.join(root, f))
                    
        print(f"Found {len(audio_files)} audio files.")
        
        with open(output_path, "w", encoding="utf-8") as out_f:
            for audio_path in audio_files:
                try:
                    audio, sr = safe_load_audio(audio_path)
                    audio = resample(audio, sr, MODEL_SAMPLE_RATE)
                    hyp = transcribe_batch_sravaani(model, [audio])[0]
                    
                    res = {"audio_path": audio_path, "hypothesis": hyp}
                    out_f.write(json.dumps(res, ensure_ascii=False) + "\n")
                    print(f"[SraVaani] {os.path.basename(audio_path)}: {hyp}")
                except Exception as e:
                    print(f"[SraVaani] Error processing {audio_path}: {e}")

if __name__ == "__main__":
    process_folder(
        dataset_dir="/kaggle/input/datasets/nagababumolleti/salesagentdata",
        manifest_path="/kaggle/working/frozen_eval_manifest.csv"
    )

Writing run_sravaani_only.py


In [12]:

%%writefile run_sravaani_only.py
import os
import json
import tempfile
import traceback
import numpy as np
import pandas as pd
import soundfile as sf
import torch

from pathlib import Path
from dataclasses import dataclass
from transformers import AutoModel
from huggingface_hub import snapshot_download


# ============================================================
# Configuration
# ============================================================

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MODEL_ID = "ARTPARK-IISc/SraVaani-1.0"
MODEL_SAMPLE_RATE = 16000

CHUNK_SECONDS = 30.0
OVERLAP_SECONDS = 1.0

DEFAULT_DATASET_DIR = (
    "/kaggle/input/datasets/nagababumolleti/salesagentdata"
)

DEFAULT_MANIFEST_PATH = "/kaggle/working/frozen_eval_manifest.csv"

DEFAULT_OUTPUT_PATH = "/kaggle/working/sravaani_results.jsonl"


@dataclass(frozen=True)
class SraVaaniChunkJob:
    file_index: int
    chunk_index: int
    audio: np.ndarray


# ============================================================
# Audio loading and resampling
# ============================================================

def safe_load_audio(path: str) -> tuple[np.ndarray, int]:
    """Load audio as mono float32 and return audio plus sample rate."""

    data, sr = sf.read(
        path,
        dtype="float32",
        always_2d=False,
    )

    # Convert stereo/multichannel audio to mono.
    if data.ndim == 2:
        data = data.mean(axis=1)

    data = np.asarray(data, dtype=np.float32).flatten()

    if data.size == 0:
        raise ValueError(f"Audio file is empty: {path}")

    if not np.all(np.isfinite(data)):
        data = np.nan_to_num(
            data,
            nan=0.0,
            posinf=0.0,
            neginf=0.0,
        )

    return data, int(sr)


def resample(
    audio: np.ndarray,
    orig_sr: int,
    target_sr: int,
) -> np.ndarray:
    """Resample audio to the model's required sample rate."""

    audio = np.asarray(audio, dtype=np.float32).flatten()

    if orig_sr <= 0:
        raise ValueError(f"Invalid original sample rate: {orig_sr}")

    if orig_sr == target_sr:
        return audio

    import librosa

    audio = librosa.resample(
        audio,
        orig_sr=orig_sr,
        target_sr=target_sr,
    )

    return np.asarray(audio, dtype=np.float32).flatten()


# ============================================================
# Model loading
# ============================================================

def build_sravaani_model(model_id: str):
    """Download and load the SraVaani model."""

    hf_token = os.environ.get("HF_TOKEN")

    print(f"Loading model: {model_id}")
    print(f"Device: {DEVICE}")

    model_dir = snapshot_download(
        repo_id=model_id,
        token=hf_token,
    )

    load_kwargs = {
        "trust_remote_code": True,
    }

    if hf_token:
        load_kwargs["token"] = hf_token

    model = AutoModel.from_pretrained(
        model_dir,
        **load_kwargs,
    )

    model = model.to(DEVICE)
    model.eval()

    print("SraVaani model loaded successfully.")

    return model


# ============================================================
# Hypothesis parsing
# ============================================================

def extract_text(hypothesis) -> str:
    """Safely extract transcript text from model output."""

    if hypothesis is None:
        return ""

    if isinstance(hypothesis, str):
        return hypothesis.strip()

    if hasattr(hypothesis, "text"):
        return str(hypothesis.text).strip()

    if isinstance(hypothesis, dict):
        for key in ("text", "transcript", "prediction"):
            if key in hypothesis:
                return extract_text(hypothesis[key])

        return ""

    if isinstance(hypothesis, (list, tuple)):
        if not hypothesis:
            return ""

        return extract_text(hypothesis[0])

    return str(hypothesis).strip()


# ============================================================
# Transcript merging
# ============================================================

def merge_texts(
    prev: str,
    curr: str,
    max_overlap_words: int = 12,
) -> str:
    """Merge adjacent chunk transcripts, removing repeated words."""

    prev = prev.strip()
    curr = curr.strip()

    if not prev:
        return curr

    if not curr:
        return prev

    if prev == curr:
        return prev

    prev_words = prev.split()
    curr_words = curr.split()

    overlap = 0

    max_n = min(
        max_overlap_words,
        len(prev_words),
        len(curr_words),
    )

    for n in range(max_n, 0, -1):
        if prev_words[-n:] == curr_words[:n]:
            overlap = n
            break

    if overlap:
        curr_words = curr_words[overlap:]

    return " ".join(prev_words + curr_words).strip()


# ============================================================
# SraVaani inference
# ============================================================

def transcribe_batch_sravaani(
    model,
    audio_list: list[np.ndarray],
    chunk_seconds: float = CHUNK_SECONDS,
    overlap_seconds: float = OVERLAP_SECONDS,
) -> list[str]:
    """
    Transcribe a list of audio arrays.

    Long recordings are split into overlapping chunks.
    Each chunk is inferred individually to help isolate
    model errors.
    """

    if not audio_list:
        return []

    chunk_samples = int(
        round(chunk_seconds * MODEL_SAMPLE_RATE)
    )

    overlap_samples = int(
        round(overlap_seconds * MODEL_SAMPLE_RATE)
    )

    step_samples = chunk_samples - overlap_samples

    if chunk_samples <= 0:
        raise ValueError("chunk_seconds must be positive.")

    if overlap_samples < 0 or step_samples <= 0:
        raise ValueError(
            "overlap_seconds must be non-negative "
            "and smaller than chunk_seconds."
        )

    jobs = []

    for file_idx, audio in enumerate(audio_list):
        audio = np.asarray(
            audio,
            dtype=np.float32,
        ).flatten()

        if audio.size == 0:
            print(f"[SraVaani] Empty audio at index {file_idx}")
            continue

        start = 0
        chunk_idx = 0

        while start < audio.size:
            end = min(
                start + chunk_samples,
                audio.size,
            )

            chunk_audio = audio[start:end]

            jobs.append(
                SraVaaniChunkJob(
                    file_index=file_idx,
                    chunk_index=chunk_idx,
                    audio=chunk_audio,
                )
            )

            chunk_idx += 1

            if end >= audio.size:
                break

            start += step_samples

    # Keep chunk indices alongside transcripts.
    file_chunks = [[] for _ in audio_list]

    if not jobs:
        return ["" for _ in audio_list]

    print(
        f"[SraVaani] Processing {len(jobs)} chunks "
        f"from {len(audio_list)} audio file(s)."
    )

    with tempfile.TemporaryDirectory(
        prefix="sravaani_chunks_"
    ) as temp_dir:

        temp_path = Path(temp_dir)

        # Write chunk audio to temporary WAV files.
        for job in jobs:
            chunk_path = (
                temp_path
                / f"file_{job.file_index}_chunk_{job.chunk_index}.wav"
            )

            sf.write(
                str(chunk_path),
                job.audio,
                MODEL_SAMPLE_RATE,
                subtype="PCM_16",
            )

        # Process each chunk separately.
        for job in jobs:
            chunk_path = (
                temp_path
                / f"file_{job.file_index}_chunk_{job.chunk_index}.wav"
            )

            try:
                with torch.inference_mode():
                    output = model.transcribe(
                        [str(chunk_path)],
                        return_hypotheses=True,
                    )

                # Handle common return formats without
                # assuming the output is a two-item tuple.
                if output is None:
                    transcript = ""

                elif isinstance(output, (list, tuple)):
                    transcript = (
                        extract_text(output[0])
                        if len(output) > 0
                        else ""
                    )

                else:
                    transcript = extract_text(output)

                file_chunks[job.file_index].append(
                    (job.chunk_index, transcript)
                )

                print(
                    f"[SraVaani] File {job.file_index}, "
                    f"chunk {job.chunk_index}: "
                    f"{transcript[:120]}"
                )

            except Exception as exc:
                print(
                    f"[SraVaani] Chunk error: "
                    f"file_index={job.file_index}, "
                    f"chunk_index={job.chunk_index}, "
                    f"error={exc}"
                )

                traceback.print_exc()

                # Retain chunk ordering if inference fails.
                file_chunks[job.file_index].append(
                    (job.chunk_index, "")
                )

    # Sort by actual chunk index, then merge transcripts.
    final_texts = []

    for chunks in file_chunks:
        chunks.sort(key=lambda item: item[0])

        merged = ""

        for _, chunk_text in chunks:
            merged = merge_texts(
                merged,
                chunk_text,
            )

        final_texts.append(merged)

    return final_texts


# ============================================================
# Manifest processing
# ============================================================

def process_folder(
    dataset_dir: str = DEFAULT_DATASET_DIR,
    manifest_path: str = DEFAULT_MANIFEST_PATH,
    output_path: str = DEFAULT_OUTPUT_PATH,
):
    """Process a manifest CSV or recursively scan a directory."""

    model = build_sravaani_model(MODEL_ID)

    output_file = Path(output_path)
    output_file.parent.mkdir(
        parents=True,
        exist_ok=True,
    )

    # --------------------------------------------------------
    # Process manifest CSV
    # --------------------------------------------------------

    if manifest_path and os.path.exists(manifest_path):
        print(f"Processing manifest: {manifest_path}")

        df = pd.read_csv(manifest_path)

        if "audio_path" not in df.columns:
            raise ValueError(
                "Manifest must contain an 'audio_path' column."
            )

        print(f"Found {len(df)} manifest rows.")

        with open(
            output_file,
            "w",
            encoding="utf-8",
        ) as out_f:

            for idx, row in df.iterrows():
                audio_path = str(row["audio_path"])

                utt_id = (
                    str(row["utterance_id"])
                    if "utterance_id" in df.columns
                    and pd.notna(row["utterance_id"])
                    else f"utt_{idx}"
                )

                reference = (
                    str(row["reference"])
                    if "reference" in df.columns
                    and pd.notna(row["reference"])
                    else ""
                )

                try:
                    if not os.path.isfile(audio_path):
                        raise FileNotFoundError(
                            f"Audio file not found: {audio_path}"
                        )

                    audio, sr = safe_load_audio(audio_path)

                    audio = resample(
                        audio,
                        sr,
                        MODEL_SAMPLE_RATE,
                    )

                    hyp = transcribe_batch_sravaani(
                        model,
                        [audio],
                    )[0]

                    result = {
                        "utterance_id": utt_id,
                        "audio_path": audio_path,
                        "hypothesis": hyp,
                        "reference": reference,
                    }

                    out_f.write(
                        json.dumps(
                            result,
                            ensure_ascii=False,
                        ) + "\n"
                    )

                    # Flush after each utterance to preserve
                    # completed results if a later row fails.
                    out_f.flush()

                    print(
                        f"[SraVaani] {utt_id}: {hyp}"
                    )

                except Exception as exc:
                    print(
                        f"[SraVaani] Error on {utt_id}: {exc}"
                    )
                    traceback.print_exc()

                    error_result = {
                        "utterance_id": utt_id,
                        "audio_path": audio_path,
                        "hypothesis": "",
                        "reference": reference,
                        "error": str(exc),
                    }

                    out_f.write(
                        json.dumps(
                            error_result,
                            ensure_ascii=False,
                        ) + "\n"
                    )

                    out_f.flush()

    # --------------------------------------------------------
    # Recursively process audio directory
    # --------------------------------------------------------

    else:
        print(
            f"Scanning directory recursively: {dataset_dir}"
        )

        if not os.path.isdir(dataset_dir):
            raise FileNotFoundError(
                f"Dataset directory not found: {dataset_dir}"
            )

        supported_extensions = (
            ".wav",
            ".mp3",
            ".flac",
            ".m4a",
            ".ogg",
            ".aac",
        )

        audio_files = sorted(
            str(path)
            for path in Path(dataset_dir).rglob("*")
            if path.is_file()
            and path.suffix.lower() in supported_extensions
        )

        print(f"Found {len(audio_files)} audio files.")

        with open(
            output_file,
            "w",
            encoding="utf-8",
        ) as out_f:

            for audio_path in audio_files:
                try:
                    audio, sr = safe_load_audio(audio_path)

                    audio = resample(
                        audio,
                        sr,
                        MODEL_SAMPLE_RATE,
                    )

                    hyp = transcribe_batch_sravaani(
                        model,
                        [audio],
                    )[0]

                    result = {
                        "audio_path": audio_path,
                        "hypothesis": hyp,
                    }

                    out_f.write(
                        json.dumps(
                            result,
                            ensure_ascii=False,
                        ) + "\n"
                    )

                    out_f.flush()

                    print(
                        f"[SraVaani] "
                        f"{os.path.basename(audio_path)}: {hyp}"
                    )

                except Exception as exc:
                    print(
                        f"[SraVaani] Error processing "
                        f"{audio_path}: {exc}"
                    )

                    traceback.print_exc()

                    error_result = {
                        "audio_path": audio_path,
                        "hypothesis": "",
                        "error": str(exc),
                    }

                    out_f.write(
                        json.dumps(
                            error_result,
                            ensure_ascii=False,
                        ) + "\n"
                    )

                    out_f.flush()

    print(f"\nResults saved to: {output_file}")


# ============================================================
# Entry point
# ============================================================

if __name__ == "__main__":
    process_folder(
        dataset_dir=DEFAULT_DATASET_DIR,
        manifest_path=DEFAULT_MANIFEST_PATH,
        output_path=DEFAULT_OUTPUT_PATH,
    )

Writing run_sravaani_only.py
